# 04 — The Precursor Test

**Do topological signatures systematically precede breakthroughs?**

This is the hypothesis test. For each cataloged breakthrough, we compare the
topological metrics (β₁, persistence entropy) in the years *before* the
breakthrough against a null model — the same CPC section pair at times when
no breakthrough occurred.

Key method: **Superposed epoch analysis** — align all breakthroughs at t=0
and average the signal. If topology is a precursor, we should see a systematic
rise before t=0 that the null model doesn't exhibit.

---

In [1]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm

from src.utils import DATA_DIR, get_logger
from src.breakthroughs import load_breakthroughs, get_precursor_window
from src.topology import compute_all_priority_pairs, ALL_PAIRS
from src.nullmodel import matched_null, random_cpc_pair_baseline, superposed_epoch
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, year_axis, confidence_band

set_style()
logger = get_logger('nb04')

CACHE_DIR = str(DATA_DIR / 'topology_cache_v2')

In [2]:
# %% Load data and breakthroughs
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

# Ensure citing_year column exists (required by new topology module)
citations['citing_year'] = pd.to_datetime(citations['citing_date']).dt.year
citations.drop(columns=['citing_date'], inplace=True)  # Free ~1GB RAM

breakthroughs = load_breakthroughs()
print(f'Breakthroughs: {len(breakthroughs)}')

2026-03-20 20:19:00 | src.breakthroughs        | INFO    | Loaded 34 breakthroughs from 8 files


Breakthroughs: 34


In [3]:
# %% Load topology results from Notebook 02 (all 28 pairs, scale-normalized)
import gc

pair_results = compute_all_priority_pairs(
    citations, cpc_map,
    cache_dir=CACHE_DIR,
    pairs=ALL_PAIRS,
    start_year=1985,
    end_year=2023,
)

# Build topology_results dict keyed by both "AxC" and "A_C" formats
topology_results = {}
for pair, group in pair_results.groupby('section_pair'):
    topology_results[pair] = group.reset_index(drop=True)
    underscore_key = pair.replace('x', '_')
    topology_results[underscore_key] = group.reset_index(drop=True)

print(f'Total topology results: {len(pair_results["section_pair"].unique())} CPC pairs (of 28)')
print(f'Total window-pair observations: {len(pair_results)}')
gc.collect()

Total topology results: 28 CPC pairs (of 28)
Total window-pair observations: 980


0

## Compute Matched Null Models

For each breakthrough, generate null topology measurements from the same
CPC pair but in non-breakthrough time periods.

In [4]:
# %% Generate matched null models for ALL breakthroughs
# With all 28 CPC pairs cached (scale-normalized), we can compute null models
# for every breakthrough — no more skipping non-priority pairs.
import pickle

null_results = {}
skipped = []
computed_fresh = []

for bt in tqdm(breakthroughs, desc='Computing null models'):
    bt_name = bt.name.replace(" ", "_").replace("/", "_").lower()[:30]
    cache_file = DATA_DIR / "null_cache" / f"matched_{bt_name}_n100_s42_cocite.pkl"
    
    if cache_file.exists():
        with open(cache_file, "rb") as f:
            null_results[bt.name] = pickle.load(f)
    else:
        # Compute fresh — should be fast with v2 topology cache
        try:
            null_df = matched_null(
                bt, citations, cpc_map,
                n_samples=100, seed=42, use_cache=True,
            )
            if len(null_df) > 0:
                null_results[bt.name] = null_df
                computed_fresh.append(bt.name)
            else:
                skipped.append(bt.name)
        except Exception as e:
            print(f'  Error for {bt.name}: {e}')
            skipped.append(bt.name)

print(f'\nNull models: {len(null_results)} breakthroughs')
print(f'  Loaded from cache: {len(null_results) - len(computed_fresh)}')
print(f'  Computed fresh: {len(computed_fresh)}')
if skipped:
    print(f'  Skipped: {skipped}')

Computing null models:   0%|          | 0/34 [00:00<?, ?it/s]

Matched null: Carbon Fiber Composites:   0%|          | 0/100 [00:00<?, ?it/s]

Matched null: Carbon Fiber Composites:  35%|███▌      | 35/100 [00:00<00:00, 347.33it/s]

Matched null: Carbon Fiber Composites:  85%|████████▌ | 85/100 [00:00<00:00, 432.44it/s]

Matched null: Carbon Fiber Composites: 100%|██████████| 100/100 [00:00<00:00, 430.62it/s]

...


2026-03-20 20:49:59 | src.nullmodel            | INFO    | Matched null for mRNA Vaccine Platform: 100/100 samples computed


2026-03-20 20:49:59 | timer                    | INFO    | matched_null completed in 0.2s


Computing null models: 100%|██████████| 34/34 [00:07<00:00,  5.18it/s]

Computing null models: 100%|██████████| 34/34 [00:07<00:00,  4.75it/s]


Null models: 34 breakthroughs
  Loaded from cache: 0
  Computed fresh: 34


## Pre-Breakthrough vs. Null Topology

For each breakthrough: extract β₁ in the 10 years before filing,
compare against the matched null distribution.

In [5]:
# %% Extract pre-breakthrough metrics and compute z-scores
# Match breakthroughs to ANY topology pair containing their CPC section(s)
comparison = []

for bt in breakthroughs:
    sections = bt.cpc_sections if len(bt.cpc_sections) >= 1 else []
    if not sections:
        continue
    
    # Find all topology pairs containing at least one of this breakthrough's sections
    matching_topos = []
    for key, topo in topology_results.items():
        if 'x' not in key:
            continue
        sec_a, sec_b = key.split('x')
        if any(s in [sec_a, sec_b] for s in sections):
            matching_topos.append(topo)
    
    if not matching_topos:
        continue
    
    year_col = 'window_end'
    start, end = get_precursor_window(bt, years_before=10)
    
    pre_beta1_vals = []
    pre_pe_vals = []
    for topo in matching_topos:
        if year_col not in topo.columns:
            continue
        pre = topo[(topo[year_col] >= start) & (topo[year_col] <= end)]
        if len(pre) > 0 and 'beta_1' in pre.columns:
            pre_beta1_vals.append(pre['beta_1'].mean())
            pre_pe_vals.append(pre['persistence_entropy'].mean())
    
    if not pre_beta1_vals:
        continue
    
    pre_beta1 = np.mean(pre_beta1_vals)
    pre_pe = np.mean(pre_pe_vals)
    
    null = null_results.get(bt.name)
    if null is None or len(null) == 0:
        continue
    
    null_beta1_mean = null['beta_1'].mean()
    null_beta1_std = null['beta_1'].std()
    null_pe_mean = null['persistence_entropy'].mean()
    null_pe_std = null['persistence_entropy'].std()
    
    # Use percentile rank only when std is truly degenerate (essentially zero)
    DEGEN_THRESH = 1e-6
    
    if null_beta1_std > DEGEN_THRESH:
        z_beta1 = (pre_beta1 - null_beta1_mean) / null_beta1_std
    else:
        pct = (null['beta_1'] < pre_beta1).mean()
        z_beta1 = stats.norm.ppf(max(0.001, min(0.999, pct)))
    
    if null_pe_std > DEGEN_THRESH:
        z_pe = (pre_pe - null_pe_mean) / null_pe_std
    else:
        pct = (null['persistence_entropy'] < pre_pe).mean()
        z_pe = stats.norm.ppf(max(0.001, min(0.999, pct)))
    
    comparison.append({
        'name': bt.name,
        'category': bt.category,
        'filing_year': bt.filing_year,
        'pre_beta1': pre_beta1,
        'null_beta1_mean': null_beta1_mean,
        'null_beta1_std': null_beta1_std,
        'z_beta1': z_beta1,
        'pre_pe': pre_pe,
        'null_pe_mean': null_pe_mean,
        'null_pe_std': null_pe_std,
        'z_pe': z_pe,
        'n_matching_pairs': len(matching_topos),
    })

comp_df = pd.DataFrame(comparison)
print(f'Breakthroughs with valid comparisons: {len(comp_df)}')
print(f'\nMean z-score (β₁):  {comp_df["z_beta1"].mean():.3f} ± {comp_df["z_beta1"].std():.3f}')
print(f'Mean z-score (PE):   {comp_df["z_pe"].mean():.3f} ± {comp_df["z_pe"].std():.3f}')
print(f'\nPer-breakthrough results:')
print(comp_df[['name', 'filing_year', 'z_beta1', 'z_pe', 'n_matching_pairs']].to_string(index=False))

Breakthroughs with valid comparisons: 21

Mean z-score (β₁):  -1.772 ± 3.298
Mean z-score (PE):   -26.020 ± 24.637

Per-breakthrough results:
                                           name  filing_year   z_beta1       z_pe  n_matching_pairs
                  Convolutional Neural Networks         1990 -2.617355 -30.830401                 7
  Hydraulic Fracturing with Horizontal Drilling         1990 -3.983011 -51.047970                 7
           Modern Wind Turbine (Variable Speed)         1990 -4.407002 -43.553254                 7
                    Elliptic Curve Cryptography         1991  3.814192   2.601041                13
                              CMOS Image Sensor         1993  3.617991   2.682361                13
                             WiFi (IEEE 802.11)         1993 -4.466650 -52.454356                 7
                       USB Universal Serial Bus         1994  3.390123   2.555507                13
                Recurrent Neural Network (LSTM)         19

## Figure 1: Superposed Epoch Plot (THE KEY RESULT)

Align all breakthroughs at t=0 (filing year), average β₁ from -10 to +5 years.
Compare against the null model's 95% confidence interval.

In [6]:
# %% Figure 1: Superposed epoch analysis
epoch = superposed_epoch(
    breakthroughs=breakthroughs,
    topology_results=topology_results,
    metric='beta_1',
    years_before=10,
    years_after=5,
)

# Generate null epoch (using random baseline)
# Use small sample — most will hit topology cache
null_baseline = random_cpc_pair_baseline(
    citations=citations,
    cpc_map=cpc_map,
    n_samples=50,
    seed=42,
)

fig, ax = plt.subplots(figsize=(12, 7))

if len(epoch) > 0:
    ax.plot(epoch['epoch_year'], epoch['mean'], color=PALETTE['red'], linewidth=2.5,
            label='Pre-breakthrough β₁ (mean)', zorder=5)
    ax.fill_between(
        epoch['epoch_year'],
        epoch['mean'] - epoch['std'],
        epoch['mean'] + epoch['std'],
        alpha=0.2, color=PALETTE['red'], label='±1 SD'
    )

# Null band
if len(null_baseline) > 0:
    null_mean = null_baseline['beta_1'].mean()
    null_std = null_baseline['beta_1'].std()
    ax.axhspan(null_mean - 1.96 * null_std, null_mean + 1.96 * null_std,
               alpha=0.1, color=PALETTE['blue'], label='Null 95% CI')
    ax.axhline(null_mean, color=PALETTE['blue'], linestyle='--', alpha=0.5)

ax.axvline(0, color='gray', linestyle=':', alpha=0.7, label='Breakthrough (t=0)')
ax.set_xlabel('Years Relative to Breakthrough Filing')
ax.set_ylabel('β₁ (Mean Across Breakthroughs)')
ax.set_title('Figure 1: Superposed Epoch Analysis — β₁ Before and After Breakthroughs')
ax.legend()
fig.tight_layout()
save_figure(fig, '04_superposed_epoch')

Random baseline:   0%|          | 0/50 [00:00<?, ?it/s]

Random baseline:  64%|██████▍   | 32/50 [00:00<00:00, 314.06it/s]

Random baseline: 100%|██████████| 50/50 [00:00<00:00, 326.03it/s]


2026-03-20 20:50:00 | src.nullmodel            | INFO    | Random baseline: 50/50 samples computed


2026-03-20 20:50:00 | timer                    | INFO    | random_cpc_pair_baseline completed in 0.2s


2026-03-20 20:50:01 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_superposed_epoch.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_superposed_epoch.png')

## Figure 2: Individual Breakthrough β₁ Curves

In [7]:
# %% Figure 2: Individual breakthrough curves
# Show a diverse selection including previously-skipped breakthroughs
selected_names = [
    'Lithium-Ion Battery', 'PageRank Algorithm',
    'CRISPR-Cas9 Gene Editing', 'mRNA Vaccine Platform',
    'Selective Laser Sintering (SLS) 3D Printing',
    'GPU General-Purpose Computing (CUDA)', 'WiFi (IEEE 802.11)',
    'CAR-T Cell Therapy', 'Convolutional Neural Networks (CNN)',
    'MapReduce Distributed Computing', 'iPhone Multi-Touch Interface',
    'Graphene Synthesis',
]

selected_bts = [bt for bt in breakthroughs if bt.name in selected_names]
# Fill with other breakthroughs if some aren't found
if len(selected_bts) < 9:
    remaining = [bt for bt in breakthroughs if bt.name not in selected_names]
    selected_bts.extend(remaining[:9 - len(selected_bts)])

n_selected = len(selected_bts)
ncols = 3
nrows = (n_selected + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows), sharex=True)
axes = axes.flatten() if n_selected > 1 else [axes]

for i, bt in enumerate(selected_bts):
    if i >= len(axes):
        break
    ax = axes[i]
    sections = bt.cpc_sections if len(bt.cpc_sections) >= 1 else []
    
    # Find all matching topology pairs
    plotted = False
    for key, topo in topology_results.items():
        if 'x' not in key:
            continue
        sec_a, sec_b = key.split('x')
        if any(s in [sec_a, sec_b] for s in sections):
            year_col = 'window_end' if 'window_end' in topo.columns else 'year'
            ax.plot(topo[year_col], topo['beta_1'], linewidth=1.5, alpha=0.6, label=key)
            plotted = True
    
    if plotted:
        ax.axvline(bt.filing_year, color=PALETTE['red'], linestyle='--', alpha=0.7, label='Filing year')
        ax.axvspan(bt.filing_year - 10, bt.filing_year, alpha=0.05, color=PALETTE['orange'])
    
    ax.set_title(bt.name, fontsize=10)
    ax.set_ylabel('β₁')
    if i == 0:
        ax.legend(fontsize=6, loc='upper right')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 2: β₁ Time Series for Individual Breakthroughs\n(colored by CPC section pair, scale-normalized distances)', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '04_individual_beta1')

2026-03-20 20:50:03 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_individual_beta1.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_individual_beta1.png')

## Figure 3: Effect Size Distribution

In [8]:
# %% Figure 3: Z-score distribution
if len(comp_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # β₁ z-scores
    ax = axes[0]
    ax.barh(range(len(comp_df)), comp_df['z_beta1'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_beta1']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3, label='p=0.05')
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (β₁)')
    ax.set_title('β₁ Effect Size')
    ax.legend()
    ax.invert_yaxis()

    # PE z-scores
    ax = axes[1]
    ax.barh(range(len(comp_df)), comp_df['z_pe'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_pe']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (Persistence Entropy)')
    ax.set_title('Persistence Entropy Effect Size')
    ax.invert_yaxis()

    fig.suptitle('Figure 3: Pre-Breakthrough Topology vs. Null Model', fontsize=14, y=1.02)
    fig.tight_layout()
    save_figure(fig, '04_effect_sizes')

2026-03-20 20:50:04 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_effect_sizes.png


## Statistical Tests

In [9]:
# %% Aggregate statistical tests
if len(comp_df) > 0:
    # One-sample t-test: are z-scores significantly different from 0?
    t_stat_b1, p_val_b1 = stats.ttest_1samp(comp_df['z_beta1'], 0)
    t_stat_pe, p_val_pe = stats.ttest_1samp(comp_df['z_pe'], 0)
    
    print('=== Aggregate Statistical Tests ===')
    print(f'N breakthroughs with valid comparisons: {len(comp_df)}')
    print(f'\nβ₁ z-scores: mean={comp_df["z_beta1"].mean():.3f}, t={t_stat_b1:.3f}, p={p_val_b1:.4f}')
    print(f'PE z-scores: mean={comp_df["z_pe"].mean():.3f}, t={t_stat_pe:.3f}, p={p_val_pe:.4f}')
    
    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat_b1, wp_b1 = stats.wilcoxon(comp_df['z_beta1'])
        w_stat_pe, wp_pe = stats.wilcoxon(comp_df['z_pe'])
        print(f'\nWilcoxon β₁: W={w_stat_b1:.1f}, p={wp_b1:.4f}')
        print(f'Wilcoxon PE: W={w_stat_pe:.1f}, p={wp_pe:.4f}')
    except ValueError as e:
        print(f'Wilcoxon test error: {e}')
        wp_b1 = wp_pe = 1.0
    
    # Holm-Bonferroni correction for multiple comparisons (4 tests)
    raw_pvals = sorted([
        ('β₁ t-test', p_val_b1),
        ('PE t-test', p_val_pe),
        ('β₁ Wilcoxon', wp_b1),
        ('PE Wilcoxon', wp_pe),
    ], key=lambda x: x[1])
    
    print(f'\n=== Holm-Bonferroni Correction (4 tests) ===')
    any_significant = False
    for i, (name, p) in enumerate(raw_pvals):
        adjusted_alpha = 0.05 / (4 - i)
        sig = '**SIGNIFICANT**' if p < adjusted_alpha else 'not significant'
        print(f'  {name}: p={p:.4f}, threshold={adjusted_alpha:.4f} → {sig}')
        if p < adjusted_alpha:
            any_significant = True
    
    # Cohen's d on raw metric values (not inflated z-scores)
    d_b1_raw = comp_df['pre_beta1'].mean() - comp_df['null_beta1_mean'].mean()
    pooled_std_b1 = np.sqrt((comp_df['pre_beta1'].std()**2 + comp_df['null_beta1_std'].mean()**2) / 2)
    d_b1 = d_b1_raw / pooled_std_b1 if pooled_std_b1 > 0 else 0
    
    d_pe_raw = comp_df['pre_pe'].mean() - comp_df['null_pe_mean'].mean()
    pooled_std_pe = np.sqrt((comp_df['pre_pe'].std()**2 + comp_df['null_pe_std'].mean()**2) / 2)
    d_pe = d_pe_raw / pooled_std_pe if pooled_std_pe > 0 else 0
    
    print(f'\nCohen\'s d (β₁, raw): {d_b1:.3f}')
    print(f'Cohen\'s d (PE, raw):  {d_pe:.3f}')
    
    # Caveats
    print(f'\n=== Caveats ===')
    print(f'- Observations are non-independent: breakthroughs share topology pairs')
    print(f'- Mean matching pairs per breakthrough: {comp_df["n_matching_pairs"].mean():.1f}')
    print(f'- Effective sample size is smaller than N={len(comp_df)}')
    print(f'- 8/34 breakthroughs skipped (non-priority CPC pairs)')
    
    # Interpretation
    if any_significant:
        print(f'\n** At least one test survives Holm-Bonferroni correction **')
    else:
        print(f'\n** No tests survive Holm-Bonferroni correction **')
        print('   Topological signatures do not systematically precede breakthroughs.')
        print('   This is a null result — itself a meaningful scientific finding.')

=== Aggregate Statistical Tests ===
N breakthroughs with valid comparisons: 21

β₁ z-scores: mean=-1.772, t=-2.462, p=0.0230
PE z-scores: mean=-26.020, t=-4.840, p=0.0001

Wilcoxon β₁: W=47.0, p=0.0158
Wilcoxon PE: W=28.0, p=0.0014

=== Holm-Bonferroni Correction (4 tests) ===
  PE t-test: p=0.0001, threshold=0.0125 → **SIGNIFICANT**
  PE Wilcoxon: p=0.0014, threshold=0.0167 → **SIGNIFICANT**
  β₁ Wilcoxon: p=0.0158, threshold=0.0250 → **SIGNIFICANT**
  β₁ t-test: p=0.0230, threshold=0.0500 → **SIGNIFICANT**

Cohen's d (β₁, raw): -5.420
Cohen's d (PE, raw):  -10.480

=== Caveats ===
- Observations are non-independent: breakthroughs share topology pairs
- Mean matching pairs per breakthrough: 9.9
- Effective sample size is smaller than N=21
- 8/34 breakthroughs skipped (non-priority CPC pairs)

** At least one test survives Holm-Bonferroni correction **


## Figure 4: ROC-Like Curve

If we treat high β₁ as a "breakthrough detector", how well does it discriminate?

In [10]:
# %% Figure 4: Detection performance
from sklearn.metrics import roc_curve, roc_auc_score

if len(comp_df) > 0 and len(null_baseline) > 0:
    # Build a proper binary classification: label real pre-breakthrough as 1, null as 0
    real_values = comp_df['pre_beta1'].values
    null_values = null_baseline['beta_1'].values
    
    all_values = np.concatenate([real_values, null_values])
    all_labels = np.concatenate([np.ones(len(real_values)), np.zeros(len(null_values))])
    
    # Use sklearn's roc_curve for correct computation
    fpr, tpr, thresholds = roc_curve(all_labels, all_values)
    auc = roc_auc_score(all_labels, all_values)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(fpr, tpr, color=PALETTE['red'], linewidth=2, label=f'β₁ detector (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC = 0.5)')
    
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Figure 4: β₁ as Breakthrough Detector')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    
    # Note the caveat
    ax.text(0.55, 0.15, f'N real = {len(real_values)}, N null = {len(null_values)}',
            fontsize=9, transform=ax.transAxes, alpha=0.6)
    
    fig.tight_layout()
    save_figure(fig, '04_roc_curve')

2026-03-20 20:50:05 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_roc_curve.png


## Summary

**The Precursor Test answers the central question:**
Do topological signatures in the patent citation network systematically
precede technological breakthroughs?

The answer depends on the data. If the superposed epoch shows a clear signal
above the null band, topology is a leading indicator. If not, innovation
is topologically invisible until it arrives — which is itself a meaningful
scientific finding.

Either way, this feeds into Notebook 05 where we test whether topology adds
predictive power beyond simpler metrics.